In [1]:
import pandas as pd
import cv2
import mediapipe as mp
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
import numpy as np

In [2]:
# Load dataset
file_path = "keypoint.csv"
df = pd.read_csv(file_path)

# Rename columns
num_features = df.shape[1]
df.columns = [f"feature_{i}" for i in range(num_features)]


In [3]:
# Assume first column is label and last column is noise
X = df.iloc[:, 1:-1]  # Exclude first (label) and last (possible noise) columns
y = df.iloc[:, 0]  # First column as target label

# Split dataset (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [5]:
# Train SVM model
svm_model = SVC(kernel='linear')
svm_model.fit(X_train, y_train)

# Predict on test set
y_pred = svm_model.predict(X_test) 
# Evaluate model accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy * 100:.2f}%")

Model Accuracy: 90.40%


In [6]:
# Real-time Hand Gesture Detection using OpenCV and Mediapipe
mp_hands = mp.solutions.hands
mp_draw = mp.solutions.drawing_utils
hands = mp_hands.Hands(min_detection_confidence=0.5, min_tracking_confidence=0.5)

In [7]:
cap = cv2.VideoCapture(0)
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    
    # Convert image to RGB
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = hands.process(rgb_frame)
    
    if results.multi_hand_landmarks:
        for hand_landmarks in results.multi_hand_landmarks:
            mp_draw.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)
            
            # Extract keypoints
            keypoints = []
            for lm in hand_landmarks.landmark:
                keypoints.append(lm.x)
                keypoints.append(lm.y)
                keypoints.append(lm.z)
            
            # Ensure correct feature length
            if len(keypoints) == X.shape[1]:
                keypoints = np.array(keypoints).reshape(1, -1)
                prediction = svm_model.predict(keypoints)[0]
                cv2.putText(frame, f"Gesture: {prediction}", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    
    cv2.imshow("Hand Gesture Recognition", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()